# Model Inference & API Serving Validation

**Course:** AIMLCZG523 - Machine Learning Operations (MLOps)

**Assignment 01 - Tasks 4 & 6:** Model Packaging, Local Inference & REST API Serving

This notebook validates local model deserialization from `models/model.joblib`, schema validation via Pydantic, and scoring of patient clinical measurements.

In [1]:
import joblib
import pandas as pd
from heart import config
from heart.api import PatientFeatures

# Load packaged pipeline (ColumnTransformer + Classifier)
pipeline = joblib.load(config.MODEL_PATH)
print(f'Successfully loaded model from {config.MODEL_PATH}')

## 1. Single Patient Prediction Demo

In [2]:
sample = {
    'age': 63.0, 'sex': 1, 'cp': 1, 'trestbps': 145.0, 'chol': 233.0,
    'fbs': 1, 'restecg': 2, 'thalach': 150.0, 'exang': 0,
    'oldpeak': 2.3, 'slope': 3, 'ca': 0.0, 'thal': 6.0
}
patient = PatientFeatures(**sample)
row = pd.DataFrame([patient.model_dump()])[config.FEATURES].astype(float)
proba = float(pipeline.predict_proba(row)[0, 1])
pred = int(proba >= 0.5)
label = 'disease' if pred == 1 else 'no_disease'
confidence = proba if pred == 1 else 1.0 - proba
print(f'Prediction: {pred} ({label})')
print(f'Probability of Heart Disease: {proba:.4f}')
print(f'Confidence: {confidence:.4f}')